## 1. What is PySpark / Spark SQL?

- **Apache Spark** is a distributed computing engine used to process very large datasets across a cluster of
  machines (or in parallel on your own laptop).
- **PySpark** is the Python API for Spark — it lets you write Spark programs in Python instead of Scala/Java.
- **Spark SQL** is the Spark module for working with *structured* data (data with rows and columns, like a
  table). It gives you two equivalent ways to work with data:
  - The **DataFrame API** — chaining Python methods like `.filter()`, `.select()`, `.groupBy()`
  - **SQL queries** — writing plain SQL strings that run against "temporary views" registered on DataFrames


In [1]:
# If PySpark isn't installed in this environment, uncomment and run the line below.
# It only needs to be run once.

# %pip install pyspark


## 2. Creating a SparkSession

The `SparkSession` is the starting point for any PySpark program. It's the object you use to create
DataFrames, register tables/views, and run SQL queries.

We create it once per notebook/session using the **builder pattern**.


In [2]:
from pyspark.sql import SparkSession

# builder.appName() just gives the Spark application a human-readable name (shows up in Spark UI / logs)
# master("local[*]") tells Spark to run locally using all available CPU cores instead of on a real cluster
# getOrCreate() reuses an existing SparkSession if one already exists, otherwise creates a new one
spark = (
    SparkSession.builder
    .appName("PySparkSQL_learn")
    .master("local[*]")
    .getOrCreate()
)

# Print the Spark version to confirm everything is working
print("Spark version:", spark.version)


Spark version: 4.0.4


## 3. Creating a DataFrame

A **DataFrame** is Spark's table-like data structure — rows of data organized into named, typed columns,
similar to a pandas DataFrame or a SQL table, but distributed across the cluster under the hood.

We'll create a small sample DataFrame of employees to use throughout this notebook.


In [3]:
# Sample data as a list of tuples — in real projects this usually comes from a file (CSV, Parquet, JSON, a database, etc.)
employee_data = [
    (1, "Asha",   "Engineering", 85000, 28),
    (2, "Rahul",  "Engineering", 92000, 34),
    (3, "Meera",  "Sales",       60000, 25),
    (4, "Zoya",   "Sales",       65000, 41),
    (5, "Vikram", "Marketing",   58000, 30),
    (6, "Ishaan", "Engineering", 78000, 23),
    (7, "Priya",  "Marketing",   62000, 36),
]

# Column names, in the same order as the tuple fields above
columns = ["id", "name", "department", "salary", "age"]

# createDataFrame() builds a DataFrame from the Python list + column names
df = spark.createDataFrame(employee_data, schema=columns)

# show() prints the DataFrame nicely as a table (truncate=False shows full column values)
df.show(truncate=False)

# printSchema() shows the inferred column names and data types
df.printSchema()


+---+------+-----------+------+---+
|id |name  |department |salary|age|
+---+------+-----------+------+---+
|1  |Asha  |Engineering|85000 |28 |
|2  |Rahul |Engineering|92000 |34 |
|3  |Meera |Sales      |60000 |25 |
|4  |Zoya  |Sales      |65000 |41 |
|5  |Vikram|Marketing  |58000 |30 |
|6  |Ishaan|Engineering|78000 |23 |
|7  |Priya |Marketing  |62000 |36 |
+---+------+-----------+------+---+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- age: long (nullable = true)



## 4. Running SQL Queries with `spark.sql()`

To query a DataFrame using SQL, you first need to register it as a **temporary view** — this gives the
DataFrame a name that SQL statements can reference, similar to a table name in a database.

- `createOrReplaceTempView("name")` — view is only visible within this SparkSession, and is replaced if it
  already exists (useful when re-running cells).


In [4]:
# Register the DataFrame as a temp view named "employees" so we can query it with SQL
df.createOrReplaceTempView("employees")

# Now we can run ordinary SQL against it using spark.sql(), which returns a new DataFrame
result = spark.sql("SELECT * FROM employees")
result.show()


+---+------+-----------+------+---+
| id|  name| department|salary|age|
+---+------+-----------+------+---+
|  1|  Asha|Engineering| 85000| 28|
|  2| Rahul|Engineering| 92000| 34|
|  3| Meera|      Sales| 60000| 25|
|  4|  Zoya|      Sales| 65000| 41|
|  5|Vikram|  Marketing| 58000| 30|
|  6|Ishaan|Engineering| 78000| 23|
|  7| Priya|  Marketing| 62000| 36|
+---+------+-----------+------+---+



## 5. Filtering with WHERE

Just like in standard SQL, `WHERE` filters rows based on a condition.

The DataFrame API equivalent is `.filter()` or `.where()` — both do the same thing.


In [5]:
# --- SQL way ---
# Select employees in the Engineering department earning more than 80000
sql_result = spark.sql('''
    SELECT name, department, salary
    FROM employees
    WHERE department = 'Engineering' AND salary > 80000
''')
sql_result.show()

# --- Equivalent DataFrame API way (for comparison) ---
df_result = (
    df.filter((df.department == "Engineering") & (df.salary > 80000))
      .select("name", "department", "salary")
)
df_result.show()


+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
| Asha|Engineering| 85000|
|Rahul|Engineering| 92000|
+-----+-----------+------+

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
| Asha|Engineering| 85000|
|Rahul|Engineering| 92000|
+-----+-----------+------+



## 6. Aggregations with GROUP BY

Aggregations summarize data — e.g., counting rows, or computing averages/sums per group.
Common SQL aggregate functions: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.


In [6]:
# For each department: count of employees, average salary, and max salary
agg_result = spark.sql('''
    SELECT
        department,
        COUNT(*)        AS num_employees,   -- number of rows in each group
        ROUND(AVG(salary), 2) AS avg_salary, -- average salary, rounded to 2 decimals
        MAX(salary)     AS max_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
''')
agg_result.show()


+-----------+-------------+----------+----------+
| department|num_employees|avg_salary|max_salary|
+-----------+-------------+----------+----------+
|Engineering|            3|   85000.0|     92000|
|      Sales|            2|   62500.0|     65000|
|  Marketing|            2|   60000.0|     62000|
+-----------+-------------+----------+----------+



## 7. Sorting with ORDER BY

`ORDER BY` sorts the result set. Use `DESC` for descending order (default is ascending).


In [8]:
# Top 3 highest-paid employees overall
top_earners = spark.sql('''
    SELECT name, department, salary
    FROM employees
    ORDER BY salary DESC
    LIMIT 3
''')
top_earners.show()


+------+-----------+------+
|  name| department|salary|
+------+-----------+------+
| Rahul|Engineering| 92000|
|  Asha|Engineering| 85000|
|Ishaan|Engineering| 78000|
+------+-----------+------+



In [11]:
# Stop the SparkSession when you're finished with it (releases resources).


# spark.stop()
print("Notebook complete. Run spark.stop() when you're done experimenting.")


Notebook complete. Run spark.stop() when you're done experimenting.
